# Section 1 — Dataset Loading and Initial Diagnosis (15 pts)

## Learning Objectives
- Load the UCI Air Quality dataset with correct parsing parameters
- Inspect structure, dtypes, and identify problematic columns
- Build a datetime column from Date and Time

## Your Decisions Log

| Step | Decision | Evidence (plot / table / stat) |
|------|----------|--------------------------------|
|      |          |                                |

## Tasks
- a) Load the dataset into a DataFrame
- b) Display the first rows
- c) Report the number of rows and columns
- d) Inspect data types of all variables
- e) Identify empty, duplicated, or unnecessary columns
- f) Convert date and time into an appropriate datetime format

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.config import RAW_DATA_PATH, FIGURES_DIR
from src.load_data import load_raw_air_quality, diagnose_structure, build_datetime

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# First, load the raw CSV exactly as it comes from the dataset.
df_raw = load_raw_air_quality(RAW_DATA_PATH)
df_raw.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,"2,6",1360.0,150.0,"11,9",1046.0,166.0,1056.0,113.0,1692.0,1268.0,"13,6","48,9","0,7578",NaN,NaN
1,10/03/2004,19.00.00,2,1292.0,112.0,"9,4",955.0,103.0,1174.0,92.0,1559.0,972.0,"13,3","47,7","0,7255",NaN,NaN
2,10/03/2004,20.00.00,"2,2",1402.0,88.0,"9,0",939.0,131.0,1140.0,114.0,1555.0,1074.0,"11,9","54,0","0,7502",NaN,NaN
3,10/03/2004,21.00.00,"2,2",1376.0,80.0,"9,2",948.0,172.0,1092.0,122.0,1584.0,1203.0,"11,0","60,0","0,7867",NaN,NaN
4,10/03/2004,22.00.00,"1,6",1272.0,51.0,"6,5",836.0,131.0,1205.0,116.0,1490.0,1110.0,"11,2","59,6","0,7888",NaN,NaN


In [ ]:
# Then inspect the dataset shape and dtypes before any cleaning.
print("=" * 72)
print("SECTION 1 - DATASET OVERVIEW")
print("=" * 72)
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")
print()

print("=" * 72)
print("DATA TYPES")
print("=" * 72)
print(df_raw.dtypes.to_string())
print()

print("=" * 72)
print("DATAFRAME INFO")
print("=" * 72)
df_raw.info()

SECTION 1 - DATASET OVERVIEW
Rows: 9,471
Columns: 17

DATA TYPES
Date                 str
Time                 str
CO(GT)               str
PT08.S1(CO)      float64
NMHC(GT)         float64
C6H6(GT)             str
PT08.S2(NMHC)    float64
NOx(GT)          float64
PT08.S3(NOx)     float64
NO2(GT)          float64
PT08.S4(NO2)     float64
PT08.S5(O3)      float64
T                    str
RH                   str
AH                   str
Unnamed: 15      float64
Unnamed: 16      float64

DATAFRAME INFO
<class 'pandas.DataFrame'>
RangeIndex: 9471 entries, 0 to 9470
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9357 non-null   str    
 1   Time           9357 non-null   str    
 2   CO(GT)         9357 non-null   str    
 3   PT08.S1(CO)    9357 non-null   float64
 4   NMHC(GT)       9357 non-null   float64
 5   C6H6(GT)       9357 non-null   str    
 6   PT08.S2(NMHC)  9357 non-null   float64
 7

In [ ]:
# This quick diagnosis highlights empty columns, duplicated names, and rows that repeat.
diagnosis = diagnose_structure(df_raw)

diagnosis_df = pd.DataFrame(
    {
        "missing_values": df_raw.isna().sum(),
        "is_empty": df_raw.isna().all(),
        "is_duplicate_name": df_raw.columns.duplicated(),
        "dtype": df_raw.dtypes.astype(str),
    }
 )

print("=" * 72)
print("DATAFRAME DIAGNOSIS")
print("=" * 72)
print(f"Duplicate rows: {diagnosis['duplicate_rows']}")
print(f"Duplicate columns: {diagnosis['duplicate_columns']}")
print(f"Empty columns: {diagnosis['empty_columns']}")
display(diagnosis_df)

DATAFRAME DIAGNOSIS


,nulls,duplicates,unnecessary
Date,114,113,False
Time,114,113,False
CO(GT),114,113,False
PT08.S1(CO),114,113,False
NMHC(GT),114,113,False
C6H6(GT),114,113,False
PT08.S2(NMHC),114,113,False
NOx(GT),114,113,False
PT08.S3(NOx),114,113,False
NO2(GT),114,113,False


In [ ]:
# Finally, create the datetime column so the time series can be analyzed as a single axis.
df = build_datetime(df_raw)
print("=" * 72)
print("DATETIME COLUMN MODIFICATION")
print("=" * 72)
display(df[['Date', 'Time', 'Datetime']].head())

DATETIME COLUMN MODIFICATION
         Date      Time            Datetime
0  10/03/2004  18.00.00 2004-03-10 18:00:00
1  10/03/2004  19.00.00 2004-03-10 19:00:00
2  10/03/2004  20.00.00 2004-03-10 20:00:00
3  10/03/2004  21.00.00 2004-03-10 21:00:00
4  10/03/2004  22.00.00 2004-03-10 22:00:00


## Guiding Questions

1. **What structural problems do you observe in the original dataset?**

   The dataset has two unnecessary columns (`Unnamed: 15` and `Unnamed: 16`) with no useful values. It also stores several numeric variables as text, and `Date` and `Time` are separated instead of being combined into a single datetime field. In addition, missing values are encoded with `-200`, so they are not immediately recognized as missing.

2. **Which variables require conversion before analysis?**

   `Date` and `Time` should be converted into a single datetime column. The variables `CO(GT)`, `C6H6(GT)`, `T`, `RH`, and `AH` should be converted from string to numeric before analysis.
